In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

def summarize_file(filename, name_col):
    df = pd.read_csv(filename)
    # Ensure timestamp exists
    if 'timestamp' in df.columns:
        # convert to UTC then to Asia/Shanghai
        ts_series = pd.to_datetime(df['timestamp'], unit='s', utc=True)
        ts_series = ts_series.dt.tz_convert(tz)
        min_ts = ts_series.min()
        max_ts = ts_series.max()
        min_ts_str = min_ts.strftime('%Y-%m-%d %H:%M:%S %Z')
        max_ts_str = max_ts.strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        min_ts_str = pd.NA
        max_ts_str = pd.NA

    total_rows = len(df)
    unique_cmdb = df['cmdb_id'].nunique() if 'cmdb_id' in df.columns else 0
    sample_cmdbs = list(pd.Series(df['cmdb_id'].dropna().unique())[:20]) if 'cmdb_id' in df.columns else []

    unique_names = df[name_col].nunique() if name_col in df and name_col in df.columns else (df[name_col].nunique() if name_col in df.columns else 0)
    sample_names = list(pd.Series(df[name_col].dropna().unique())[:20]) if name_col in df.columns else []

    summary = {
        'file': filename,
        'total_rows': total_rows,
        'min_timestamp_UTC+8': min_ts_str,
        'max_timestamp_UTC+8': max_ts_str,
        'unique_cmdb_count': unique_cmdb,
        'sample_cmdb_ids_up_to_20': sample_cmdbs,
        'unique_name_count': unique_names,
        'sample_names_up_to_20': sample_names
    }
    return df, summary

# Load and summarize each file
metric_df, metric_summary = summarize_file('metric.csv', 'kpi_name')
trace_df, trace_summary = summarize_file('trace.csv', 'trace_name')
log_df, log_summary = summarize_file('log.csv', 'log_name')
error_df, error_summary = summarize_file('error_logs.csv', 'message')

# Compose a compact DataFrame of summaries
summary = pd.DataFrame([metric_summary, trace_summary, log_summary, error_summary])[
    ['file','total_rows','min_timestamp_UTC+8','max_timestamp_UTC+8',
     'unique_cmdb_count','sample_cmdb_ids_up_to_20',
     'unique_name_count','sample_names_up_to_20']
]

summary

```
Out[1]:
```
```python
# Summarize the previously computed summaries in plain English (compact)
def _short(lst, n=6):
    try:
        return ', '.join(lst[:n]) + (', ...' if len(lst) > n else '')
    except Exception:
        return str(lst)

metric = metric_summary
trace = trace_summary
log = log_summary
err = error_summary

summary_text = (
    "Files summary (times shown in UTC+8):\n"
    f"- metric.csv: {metric['total_rows']} rows; time range {metric['min_timestamp_UTC+8']} to {metric['max_timestamp_UTC+8']}; "
    f"{metric['unique_cmdb_count']} unique cmdb_id (sample: {_short(metric['sample_cmdb_ids_up_to_20'])}); "
    f"{metric['unique_name_count']} unique KPI names (sample: {_short(metric['sample_names_up_to_20'])}).\n"
    f"- trace.csv: {trace['total_rows']} rows; time range {trace['min_timestamp_UTC+8']} to {trace['max_timestamp_UTC+8']}; "
    f"{trace['unique_cmdb_count']} unique cmdb_id (sample: {_short(trace['sample_cmdb_ids_up_to_20'])}); "
    f"{trace['unique_name_count']} unique trace names (sample: {_short(trace['sample_names_up_to_20'])}).\n"
    f"- log.csv: {log['total_rows']} rows; time range {log['min_timestamp_UTC+8']} to {log['max_timestamp_UTC+8']}; "
    f"{log['unique_cmdb_count']} unique cmdb_id (sample: {_short(log['sample_cmdb_ids_up_to_20'])}); "
    f"{log['unique_name_count']} unique log names (sample: {_short(log['sample_names_up_to_20'])}).\n"
    f"- error_logs.csv: {err['total_rows']} rows; time range {err['min_timestamp_UTC+8']} to {err['max_timestamp_UTC+8']}; "
    f"{err['unique_cmdb_count']} unique cmdb_id (sample: {_short(err['sample_cmdb_ids_up_to_20'])}); "
    f"{err['unique_name_count']} unique message variants (sample: {_short(err['sample_names_up_to_20'])})."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

file  total_rows      min_timestamp_UTC+8      max_timestamp_UTC+8  unique_cmdb_count                           sample_cmdb_ids_up_to_20  unique_name_count                              sample_names_up_to_20
0      metric.csv      145092  2022-03-20 19:00:00 CST  2022-03-20 19:29:00 CST                 59  [adservice, adservice-0, adservice-1, adservic...               1176  [app.grpc.count, app.grpc.mrt, app.grpc.rr, ap...
1       trace.csv       36948  2022-03-20 19:00:00 CST  2022-03-20 19:29:00 CST                 40  [adservice-0, adservice-1, adservice-2, adserv...                200  [trace.from_frontend-0.duration_mean, trace.fr...
2         log.csv        1814  2022-03-20 19:00:00 CST  2022-03-20 19:29:00 CST                 31  [adservice-0, adservice-1, adservice-2, cartse...                  2                   [log.error_count, log.row_count]
3  error_logs.csv        2388  2022-03-20 19:00:00 CST  2022-03-20 19:29:59 CST                  3               [frontend-0, frontend-2, frontend-1]                  1  [severity: warning, message: failed to retriev...```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')

# 1) Load metric.csv if not already present
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure timestamp as timezone-aware datetimes in UTC+8 and keep original numeric timestamp if needed
metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 2) Compute global P95 and P5 for each (cmdb_id, kpi_name) using the full series (no time filtering)
grp = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
global_p95 = grp.quantile(0.95).rename('global_P95')
global_p5 = grp.quantile(0.05).rename('global_P5')
global_pct = pd.concat([global_p95, global_p5], axis=1).reset_index()

# 3) Filter metric.csv to the incident window: 2022-03-20 19:00:00 to 2022-03-20 19:30:00 UTC+8 (inclusive start, up to 19:29:59)
start_dt = tz.localize(datetime(2022, 3, 20, 19, 0, 0))
end_dt = tz.localize(datetime(2022, 3, 20, 19, 30, 0))  # exclusive upper bound
window_df = metric_df[(metric_df['ts'] >= start_dt) & (metric_df['ts'] < end_dt)].copy()

# 4) Merge global percentiles into the windowed records
merged = window_df.merge(global_pct, on=['cmdb_id', 'kpi_name'], how='left')

# Flags for anomalies relative to global thresholds
merged['is_high'] = merged['value'] >= merged['global_P95']
merged['is_low'] = merged['value'] <= merged['global_P5']

# Helper timestamp columns for earliest anomaly detection
merged['ts_high'] = merged['ts'].where(merged['is_high'])
merged['ts_low'] = merged['ts'].where(merged['is_low'])

# 5) Aggregate per (cmdb_id, kpi_name) within the window
agg_window = merged.groupby(['cmdb_id', 'kpi_name']).agg(
    global_P95=('global_P95', 'first'),
    global_P5=('global_P5', 'first'),
    total_points_in_window=('value', 'size'),
    max_value_in_window=('value', 'max'),
    min_value_in_window=('value', 'min'),
    anomaly_count_high=('is_high', 'sum'),
    anomaly_count_low=('is_low', 'sum'),
    earliest_anomaly_ts_high=('ts_high', 'min'),
    earliest_anomaly_ts_low=('ts_low', 'min'),
).reset_index()

# 6) Ensure combos that did NOT appear in the window are accounted for (with zeros)
all_combos = global_pct.copy()
agg_full = all_combos.merge(
    agg_window,
    on=['cmdb_id', 'kpi_name', 'global_P95', 'global_P5'],
    how='left',
    suffixes=('', '_w')
)

# Fill missing window-derived fields with zeros / NA
agg_full['total_points_in_window'] = agg_full['total_points_in_window'].fillna(0).astype(int)
agg_full['max_value_in_window'] = agg_full['max_value_in_window']  # keep NaN if no points
agg_full['min_value_in_window'] = agg_full['min_value_in_window']
agg_full['anomaly_count_high'] = agg_full['anomaly_count_high'].fillna(0).astype(int)
agg_full['anomaly_count_low'] = agg_full['anomaly_count_low'].fillna(0).astype(int)
agg_full['earliest_anomaly_ts_high'] = pd.to_datetime(agg_full['earliest_anomaly_ts_high']).dt.tz_convert(tz)
agg_full['earliest_anomaly_ts_low'] = pd.to_datetime(agg_full['earliest_anomaly_ts_low']).dt.tz_convert(tz)

# Format earliest timestamps as compact strings, keep nulls as <NA>
agg_full['earliest_anomaly_ts_high'] = agg_full['earliest_anomaly_ts_high'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
agg_full['earliest_anomaly_ts_low'] = agg_full['earliest_anomaly_ts_low'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
agg_full['earliest_anomaly_ts_high'] = agg_full['earliest_anomaly_ts_high'].where(agg_full['earliest_anomaly_ts_high'].notna(), pd.NA)
agg_full['earliest_anomaly_ts_low'] = agg_full['earliest_anomaly_ts_low'].where(agg_full['earliest_anomaly_ts_low'].notna(), pd.NA)

# 7) Add anomaly total for sorting and produce the top-20 table
agg_full['anomaly_total'] = agg_full['anomaly_count_high'] + agg_full['anomaly_count_low']
selected_cols = [
    'cmdb_id', 'kpi_name', 'global_P95', 'global_P5',
    'total_points_in_window', 'anomaly_count_high', 'earliest_anomaly_ts_high',
    'anomaly_count_low', 'earliest_anomaly_ts_low',
    'max_value_in_window', 'min_value_in_window', 'anomaly_total'
]
agg_sorted = agg_full.sort_values(by='anomaly_total', ascending=False).reset_index(drop=True)

# Top 20 rows (compact)
result_table = agg_sorted[selected_cols].head(20)

# 8) One-line summary: total distinct combos scanned and how many had anomalies in the incident window
total_scanned = all_combos.shape[0]
num_with_anomaly = int((agg_full['anomaly_total'] > 0).sum())
summary_line = f"{total_scanned} distinct (cmdb_id,kpi_name) scanned; {num_with_anomaly} had anomaly_count_high+anomaly_count_low > 0 in the incident window."

# Keep full intermediate variables in kernel for later steps:
# global_pct, window_df, merged, agg_window, agg_full, agg_sorted

result_table, summary_line

```
Out[2]:
```
```python
# Compose a plain-English compact summary using previously computed results (reuse kernel variables)
top_cmdbs = result_table['cmdb_id'].unique().tolist()
top_sample = ", ".join(top_cmdbs[:3]) + (", ..." if len(top_cmdbs) > 3 else "")

summary = (
    f"Scanned {total_scanned} distinct (cmdb_id,kpi_name) combinations; "
    f"{num_with_anomaly} had anomaly_count_high+anomaly_count_low > 0 in the 2022-03-20 19:00–19:29:59 UTC+8 window. "
    f"The top anomalies are dominated by {top_sample} (example: shippingservice2-0) where many container filesystem and memory KPIs "
    f"have anomaly_count_high=30 and anomaly_count_low=30 over 30 points (anomaly_total=60), with earliest anomaly at 2022-03-20 19:00:00 UTC+8. "
    "In short: every scanned metric reported anomalies in the incident window; shippingservice2-0 metrics are the most frequent top entries."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                           kpi_name    global_P95     global_P5  total_points_in_window  anomaly_count_high earliest_anomaly_ts_high  anomaly_count_low  earliest_anomaly_ts_low  max_value_in_window  min_value_in_window  anomaly_total
0   shippingservice2-0  container.node-5.container_fs_writes_merged./d...  0.000000e+00  0.000000e+00                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         0.000000e+00         0.000000e+00             60
1   shippingservice2-0            container.node-5.container_memory_cache  1.854669e+07  1.854669e+07                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         1.854669e+07         1.854669e+07             60
2   shippingservice2-0          container.node-5.container_memory_failcnt  2.397029e+06  2.397029e+06                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         2.397029e+06         2.397029e+06             60
3   shippingservice2-0  container.node-5.container_memory_failures.con...  0.000000e+00  0.000000e+00                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         0.000000e+00         0.000000e+00             60
4   shippingservice2-0  container.node-5.container_memory_failures.hie...  0.000000e+00  0.000000e+00                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         0.000000e+00         0.000000e+00             60
5   shippingservice2-0  container.node-5.container_fs_reads_merged./de...  0.000000e+00  0.000000e+00                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         0.000000e+00         0.000000e+00             60
6   shippingservice2-0  container.node-5.container_fs_sector_reads./de...  0.000000e+00  0.000000e+00                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         0.000000e+00         0.000000e+00             60
7   shippingservice2-0  container.node-5.container_fs_sector_writes./d...  0.000000e+00  0.000000e+00                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         0.000000e+00         0.000000e+00             60
8   shippingservice2-0  container.node-5.container_fs_write_seconds./d...  0.000000e+00  0.000000e+00                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         0.000000e+00         0.000000e+00             60
9   shippingservice2-0      container.node-5.container_fs_writes./dev/vda  0.000000e+00  0.000000e+00                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         0.000000e+00         0.000000e+00             60
10  shippingservice2-0     container.node-5.container_fs_writes./dev/vda1  0.000000e+00  0.000000e+00                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         0.000000e+00         0.000000e+00             60
11  shippingservice2-0   container.node-5.container_fs_writes_MB./dev/vda  0.000000e+00  0.000000e+00                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         0.000000e+00         0.000000e+00             60
12  shippingservice2-0  container.node-5.container_fs_io_current./dev/...  0.000000e+00  0.000000e+00                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         0.000000e+00         0.000000e+00             60
13  shippingservice2-0  container.node-5.container_fs_io_time_seconds....  0.000000e+00  0.000000e+00                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         0.000000e+00         0.000000e+00             60
14  shippingservice2-0  container.node-5.container_fs_io_time_weighted...  0.000000e+00  0.000000e+00                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         0.000000e+00         0.000000e+00             60
15  shippingservice2-0   container.node-5.container_fs_limit_MB./dev/vda1  6.046307e+05  6.046307e+05                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         6.046307e+05         6.046307e+05             60
16  shippingservice2-0  container.node-5.container_fs_read_seconds./de...  0.000000e+00  0.000000e+00                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         0.000000e+00         0.000000e+00             60
17  shippingservice2-0       container.node-5.container_fs_reads./dev/vda  0.000000e+00  0.000000e+00                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         0.000000e+00         0.000000e+00             60
18  shippingservice2-0      container.node-5.container_fs_reads./dev/vda1  0.000000e+00  0.000000e+00                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         0.000000e+00         0.000000e+00             60
19  shippingservice2-0    container.node-5.container_fs_reads_MB./dev/vda  0.000000e+00  0.000000e+00                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST         0.000000e+00         0.000000e+00             60, '4846 distinct (cmdb_id,kpi_name) scanned; 4846 had anomaly_count_high+anomaly_count_low > 0 in the incident window.')```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse agg_full, result_table, total_scanned from kernel state
# agg_full should contain all combos with columns used earlier including global_P95, global_P5, max_value_in_window, min_value_in_window, anomaly_total, anomaly_count_high/low, earliest_anomaly_ts_*

# Compute safe denominators
denom_high = np.maximum(1e-9, agg_full['global_P95'].abs())
denom_low  = np.maximum(1e-9, agg_full['global_P5'].abs())

# For combos with no points, max/min may be NaN — replace with the corresponding global threshold so computed breach is zero
max_vals = agg_full['max_value_in_window'].fillna(agg_full['global_P95'])
min_vals = agg_full['min_value_in_window'].fillna(agg_full['global_P5'])

# Compute severities
agg_full['severity_high'] = (max_vals - agg_full['global_P95']) / denom_high
agg_full['severity_low']  = (agg_full['global_P5'] - min_vals) / denom_low

# Replace infinities/nans (if any) with 0 to be safe
agg_full['severity_high'].replace([np.inf, -np.inf], 0, inplace=True)
agg_full['severity_low'].replace([np.inf, -np.inf], 0, inplace=True)
agg_full['severity_high'].fillna(0, inplace=True)
agg_full['severity_low'].fillna(0, inplace=True)

# severity is the max of the two
agg_full['severity'] = agg_full[['severity_high', 'severity_low']].max(axis=1)

# 1) Filter rows where anomaly_total > 0 and severity > 0.5
severity_candidates = agg_full[(agg_full['anomaly_total'] > 0) & (agg_full['severity'] > 0.5)].copy()

# 2) Return up to top 20 rows sorted by severity descending with requested columns
out_cols = [
    'cmdb_id', 'kpi_name', 'global_P95', 'global_P5', 'total_points_in_window',
    'anomaly_count_high', 'earliest_anomaly_ts_high', 'anomaly_count_low', 'earliest_anomaly_ts_low',
    'max_value_in_window', 'min_value_in_window', 'anomaly_total', 'severity'
]

severity_candidates_top20 = severity_candidates.sort_values(by='severity', ascending=False)[out_cols].head(20)

# 3) One-line aggregate: total scanned (reuse total_scanned) and how many have severity>0.5
num_sev_gt_05 = int((agg_full['severity'] > 0.5).sum())
summary_line = f"{total_scanned} distinct (cmdb_id,kpi_name) scanned; {num_sev_gt_05} have severity > 0.5."

# Display compact results (DataFrame head and one-line summary)
severity_candidates_top20, summary_line

```
Out[3]:
```
# Compose a compact plain-English summary string using previously computed variables
try:
    total = total_scanned
except NameError:
    total = int(agg_full.shape[0])

try:
    num_sev = num_sev_gt_05
except NameError:
    num_sev = int((agg_full['severity'] > 0.5).sum())

# Build a short sample of top severe candidates (up to 5) from severity_candidates_top20 if available
try:
    sample_df = severity_candidates_top20[['cmdb_id','kpi_name','severity']].head(5)
    samples = []
    for _, r in sample_df.iterrows():
        samples.append(f"{r['cmdb_id']} | {r['kpi_name']} (sev={r['severity']:.3e})")
    sample_text = "; ".join(samples)
    if sample_text:
        sample_text = " Top examples: " + sample_text + "."
    else:
        sample_text = ""
except Exception:
    sample_text = ""

summary_text = (
    f"Scanned {total} distinct (cmdb_id,kpi_name) combinations; "
    f"{num_sev} have severity > 0.5 in the 2022-03-20 19:00–19:29:59 UTC+8 window. "
    "Many of the highest-severity items are container-level metrics where the global threshold (P95 or P5) is zero "
    "and the window value is non-zero (leading to very large computed severities). "
    "Top affected components include cartservice pods (CPU throttling metrics), paymentservice2 (memory failure metrics), "
    "and several services showing elevated network receive MB metrics with moderate severity." 
    + sample_text
)

summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name  global_P95  global_P5  total_points_in_window  anomaly_count_high earliest_anomaly_ts_high  anomaly_count_low  earliest_anomaly_ts_low  max_value_in_window  min_value_in_window  anomaly_total      severity
1031       cartservice-0  container.node-5.container_cpu_cfs_throttled_p...    0.000000   0.000000                      30                  30  2022-03-20 19:00:00 CST                 29  2022-03-20 19:00:00 CST             2.000000             0.000000             59  2.000000e+09
1119       cartservice-1  container.node-5.container_cpu_cfs_throttled_p...    0.000000   0.000000                      30                  30  2022-03-20 19:00:00 CST                 29  2022-03-20 19:00:00 CST             2.000000             0.000000             59  2.000000e+09
3691   paymentservice2-0  container.node-6.container_memory_failures.con...    0.000000   0.000000                      30                  30  2022-03-20 19:00:00 CST                 29  2022-03-20 19:00:00 CST             1.000000             0.000000             59  1.000000e+09
1863   currencyservice-0  container.node-5.container_cpu_cfs_throttled_p...    0.000000   0.000000                      30                  30  2022-03-20 19:00:00 CST                 29  2022-03-20 19:00:00 CST             1.000000             0.000000             59  1.000000e+09
3693   paymentservice2-0  container.node-6.container_memory_failures.hie...    0.000000   0.000000                      30                  30  2022-03-20 19:00:00 CST                 29  2022-03-20 19:00:00 CST             1.000000             0.000000             59  1.000000e+09
3675   paymentservice2-0       container.node-6.container_fs_reads./dev/vda    0.000000   0.000000                      30                  30  2022-03-20 19:00:00 CST                 29  2022-03-20 19:00:00 CST             1.000000             0.000000             59  1.000000e+09
818           adservice2  runtime.java_lang_Memory_ObjectPendingFinaliza...    0.000000   0.000000                      30                  30  2022-03-20 19:00:00 CST                 29  2022-03-20 19:00:00 CST             0.750000             0.000000             59  7.500000e+08
1120       cartservice-1  container.node-5.container_cpu_cfs_throttled_s...    0.000000   0.000000                      30                  30  2022-03-20 19:00:00 CST                 29  2022-03-20 19:00:00 CST             0.139845             0.000000             59  1.398450e+08
1032       cartservice-0  container.node-5.container_cpu_cfs_throttled_s...    0.000000   0.000000                      30                  30  2022-03-20 19:00:00 CST                 29  2022-03-20 19:00:00 CST             0.091457             0.000000             59  9.145650e+07
1864   currencyservice-0  container.node-5.container_cpu_cfs_throttled_s...    0.000000   0.000000                      30                  30  2022-03-20 19:00:00 CST                 29  2022-03-20 19:00:00 CST             0.031301             0.000000             59  3.130054e+07
3677   paymentservice2-0    container.node-6.container_fs_reads_MB./dev/vda    0.000000   0.000000                      30                  30  2022-03-20 19:00:00 CST                 29  2022-03-20 19:00:00 CST             0.007812             0.000000             59  7.812500e+06
2472          frontend-0  container.node-5.container_cpu_cfs_throttled_s...    0.000000   0.000000                      30                  30  2022-03-20 19:00:00 CST                 29  2022-03-20 19:00:00 CST             0.001512             0.000000             59  1.512169e+06
2315      emailservice-1  mesh.source.emailservice.jaeger-collector.isti...    0.000000   0.000000                      30                  30  2022-03-20 19:00:00 CST                 29  2022-03-20 19:00:00 CST             0.001050             0.000000             59  1.050000e+06
2788          frontend-2  container.node-5.container_cpu_cfs_throttled_s...    0.000000   0.000000                      30                  30  2022-03-20 19:00:00 CST                 29  2022-03-20 19:00:00 CST             0.000894             0.000000             59  8.938515e+05
3494    paymentservice-0  container.node-5.container_network_receive_MB....    0.025556   0.018842                      30                   2  2022-03-20 19:11:00 CST                  2  2022-03-20 19:03:00 CST             0.554951             0.017567              4  2.071473e+01
3700   paymentservice2-0  container.node-6.container_network_receive_MB....    0.027240   0.017185                      30                   2  2022-03-20 19:07:00 CST                  2  2022-03-20 19:02:00 CST             0.554198             0.013413              4  1.934478e+01
2210      emailservice-0  container.node-5.container_network_receive_MB....    0.031563   0.021508                      30                   2  2022-03-20 19:09:00 CST                  2  2022-03-20 19:12:00 CST             0.566823             0.020815              4  1.695828e+01
4806  shippingservice2-0  container.node-5.container_network_receive_MB....    0.044631   0.029403                      30                   3  2022-03-20 19:00:00 CST                  2  2022-03-20 19:02:00 CST             0.567022             0.026243              5  1.170466e+01
4646   shippingservice-1  container.node-5.container_network_receive_MB....    0.046914   0.038991                      30                   2  2022-03-20 19:14:00 CST                  2  2022-03-20 19:03:00 CST             0.578275             0.037670              4  1.132621e+01
4726   shippingservice-2  container.node-5.container_network_receive_MB....    0.049310   0.036692                      30                   2  2022-03-20 19:06:00 CST                  2  2022-03-20 19:05:00 CST             0.568259             0.034569              4  1.052428e+01, '4846 distinct (cmdb_id,kpi_name) scanned; 134 have severity > 0.5.')```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse severity_candidates from kernel; ensure tz exists
tz = pytz.timezone('Asia/Shanghai')

# If severity_candidates not present or empty, create an empty DataFrame to avoid errors
try:
    severity_candidates
except NameError:
    severity_candidates = pd.DataFrame()

# Work on a copy to avoid mutating original
sc = severity_candidates.copy()

# Prepare timestamp parsing: remove trailing timezone name (like ' CST') then parse
for col in ['earliest_anomaly_ts_high', 'earliest_anomaly_ts_low']:
    if col in sc.columns:
        # Normalize strings: drop trailing timezone tokens (e.g., ' CST') to parse reliably, coerce errors to NaT
        parsed = pd.to_datetime(
            sc[col].astype(str).str.replace(r'\s+[A-Z]+$', '', regex=True),
            errors='coerce',
            format='%Y-%m-%d %H:%M:%S'
        )
        # Localize naive datetimes to Asia/Shanghai (UTC+8)
        # If parsed is dtype datetime64[ns], then it's naive -> tz_localize
        parsed = parsed.dt.tz_localize(tz)
        sc[col + '_parsed'] = parsed
    else:
        sc[col + '_parsed'] = pd.NaT

# Per-row earliest severity timestamp (min of high/low parsed)
sc['earliest_row_severity_ts'] = sc[['earliest_anomaly_ts_high_parsed', 'earliest_anomaly_ts_low_parsed']].min(axis=1)

# Now aggregate by cmdb_id
if sc.shape[0] == 0:
    component_candidates = pd.DataFrame(columns=[
        'cmdb_id', 'num_severe_kpis', 'max_severity', 'mean_severity',
        'earliest_severity_ts', 'total_anomaly_kpis', 'sample_kpis'
    ])
else:
    # For sample_kpis, pick up to 5 kpi_name values with highest severity per cmdb_id
    def top_kpis(series_df):
        # series_df is a DataFrame for the group
        top = series_df.sort_values('severity', ascending=False).head(5)['kpi_name'].tolist()
        return "; ".join(top)

    grp = sc.groupby('cmdb_id')

    comp_agg = grp.agg(
        num_severe_kpis=('severity', 'size'),
        max_severity=('severity', 'max'),
        mean_severity=('severity', 'mean'),
        total_anomaly_kpis=('anomaly_total', 'sum'),
        earliest_severity_ts_parsed=('earliest_row_severity_ts', 'min')
    ).reset_index()

    # Compute sample_kpis separately using apply to preserve ordering by severity
    sample_kpis = grp.apply(top_kpis).reset_index().rename(columns={0: 'sample_kpis'})

    component_candidates = comp_agg.merge(sample_kpis, on='cmdb_id', how='left')

    # Format earliest_severity_ts as string in UTC+8
    component_candidates['earliest_severity_ts'] = component_candidates['earliest_severity_ts_parsed'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    component_candidates['earliest_severity_ts'] = component_candidates['earliest_severity_ts'].where(component_candidates['earliest_severity_ts'].notna(), pd.NA)

    # Keep only requested columns and sort
    component_candidates = component_candidates[
        ['cmdb_id', 'num_severe_kpis', 'max_severity', 'mean_severity',
         'earliest_severity_ts', 'total_anomaly_kpis', 'sample_kpis']
    ].sort_values(by=['num_severe_kpis', 'max_severity'], ascending=[False, False]).reset_index(drop=True)

# Store for next steps
# component_candidates variable is now available in the kernel

# Top 10 compact table
component_candidates_top10 = component_candidates.head(10)

# One-line summary: total distinct cmdb_id in severity_candidates and how many have num_severe_kpis >= 3
total_distinct_cmdb = int(component_candidates.shape[0])
num_with_at_least_3 = int((component_candidates['num_severe_kpis'] >= 3).sum()) if total_distinct_cmdb > 0 else 0
summary_line = f"{total_distinct_cmdb} distinct cmdb_id in severity_candidates; {num_with_at_least_3} have num_severe_kpis >= 3."

# Display compact outputs (DataFrame head and summary)
component_candidates_top10, summary_line

```
Out[4]:
```
Summary (plain English, compact):

- In total 50 distinct components (cmdb_id) were identified among severity candidates; 18 of them have >= 3 severe KPIs (severity > 0.5).

- Top aggregated candidates (sorted by number of severe KPIs, then severity):
  1. paymentservice2-0 — num_severe_kpis=9, max_severity=1.0e+09, mean_severity≈3.34e+08, earliest=2022-03-20 19:00:00 UTC+8, total_anomaly_kpis=258.
  2. node-2 — num_severe_kpis=9, max_severity≈4.14, mean_severity≈1.48, earliest=2022-03-20 19:00:00 UTC+8, total_anomaly_kpis=103.
  3. adservice2-0 — num_severe_kpis=8, max_severity≈6.19, mean_severity≈2.73, earliest=2022-03-20 19:00:00 UTC+8, total_anomaly_kpis=74.
  4. node-1 — num_severe_kpis=7, max_severity≈2.21, mean_severity≈1.30, earliest=2022-03-20 19:01:00 UTC+8, total_anomaly_kpis=47.
  5. node-6 — num_severe_kpis=7, max_severity≈1.68, mean_severity≈1.08, earliest=2022-03-20 19:00:00 UTC+8, total_anomaly_kpis=50.
  6. cartservice-1 — num_severe_kpis=5, max_severity=2.0e+09, mean_severity≈4.28e+08, earliest=2022-03-20 19:00:00 UTC+8, total_anomaly_kpis=131.
  7. shippingservice-1 — num_severe_kpis=5, max_severity≈11.33, mean_severity≈2.87, earliest=2022-03-20 19:00:00 UTC+8, total_anomaly_kpis=54.
  8. shippingservice-2 — num_severe_kpis=5, max_severity≈10.52, mean_severity≈2.90, earliest=2022-03-20 19:00:00 UTC+8, total_anomaly_kpis=47.
  9. node-5 — num_severe_kpis=5, max_severity≈3.56, mean_severity≈1.60, earliest=2022-03-20 19:00:00 UTC+8, total_anomaly_kpis=111.
 10. cartservice-0 — num_severe_kpis=4, max_severity=2.0e+09, mean_severity≈5.23e+08, earliest=2022-03-20 19:00:00 UTC+8, total_anomaly_kpis=126.

- Note: Several very large severities (e.g., 1e9 or 2e9) result from global thresholds equal to zero while window values are non-zero (leading to very large normalized breach scores). These component-level aggregates and sample KPIs are stored in component_candidates for the next inspection step.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id  num_severe_kpis  max_severity  mean_severity     earliest_severity_ts  total_anomaly_kpis                                        sample_kpis
0  paymentservice2-0                9  1.000000e+09   3.342014e+08  2022-03-20 19:00:00 CST                 258  container.node-6.container_fs_reads./dev/vda; ...
1             node-2                9  4.135135e+00   1.481481e+00  2022-03-20 19:00:00 CST                 103  system.io.avg_q_sz; system.io.w_await; system....
2       adservice2-0                8  6.186771e+00   2.728345e+00  2022-03-20 19:00:00 CST                  74  container.node-6.container_network_receive_MB....
3             node-1                7  2.206897e+00   1.296205e+00  2022-03-20 19:01:00 CST                  47  system.net.tcp.retrans_segs; system.cpu.user; ...
4             node-6                7  1.682927e+00   1.083103e+00  2022-03-20 19:00:00 CST                  50  system.net.tcp.retrans_segs; system.net.tcp.ou...
5      cartservice-1                5  2.000000e+09   4.279690e+08  2022-03-20 19:00:00 CST                 131  container.node-5.container_cpu_cfs_throttled_p...
6  shippingservice-1                5  1.132621e+01   2.874083e+00  2022-03-20 19:00:00 CST                  54  container.node-5.container_network_receive_MB....
7  shippingservice-2                5  1.052428e+01   2.895644e+00  2022-03-20 19:00:00 CST                  47  container.node-5.container_network_receive_MB....
8             node-5                5  3.561404e+00   1.596044e+00  2022-03-20 19:00:00 CST                 111  system.io.avg_q_sz; system.io.w_s; system.io.u...
9      cartservice-0                4  2.000000e+09   5.228641e+08  2022-03-20 19:00:00 CST                 126  container.node-5.container_cpu_cfs_throttled_p..., '50 distinct cmdb_id in severity_candidates; 18 have num_severe_kpis >= 3.')```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_df if present, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Ensure timestamp column as timezone-aware Asia/Shanghai datetimes
trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Prepare top-10 cmdb_id from component_candidates (already in kernel)
try:
    component_candidates
except NameError:
    # fallback: compute an empty DataFrame
    component_candidates = pd.DataFrame(columns=['cmdb_id','num_severe_kpis'])

top_cmdbs = component_candidates.sort_values(['num_severe_kpis','max_severity'], ascending=[False, False])['cmdb_id'].head(10).tolist()

# 1) Compute global P95 and P5 for every (cmdb_id, trace_name) using full series
grp_trace = trace_df.groupby(['cmdb_id', 'trace_name'])['value']
trace_global_p95 = grp_trace.quantile(0.95).rename('global_P95')
trace_global_p5  = grp_trace.quantile(0.05).rename('global_P5')
trace_thresholds = pd.concat([trace_global_p95, trace_global_p5], axis=1).reset_index()

# 2) Filter trace.csv to incident window (inclusive 19:00, exclusive 19:30)
start_dt = tz.localize(datetime(2022,3,20,19,0,0))
end_dt   = tz.localize(datetime(2022,3,20,19,30,0))
window_trace = trace_df[(trace_df['ts'] >= start_dt) & (trace_df['ts'] < end_dt)].copy()

# Further limit to top_cmdbs for analysis
window_top = window_trace[window_trace['cmdb_id'].isin(top_cmdbs)].copy()

# 3) Merge global thresholds into the windowed records
window_top = window_top.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

# Flags and earliest ts
window_top['is_high'] = window_top['value'] >= window_top['global_P95']
window_top['is_low']  = window_top['value'] <= window_top['global_P5']
window_top['ts_high'] = window_top['ts'].where(window_top['is_high'])
window_top['ts_low']  = window_top['ts'].where(window_top['is_low'])

# Aggregate per (cmdb_id, trace_name) within the window
trace_detail_df = window_top.groupby(['cmdb_id','trace_name']).agg(
    total_points_in_window = ('value','size'),
    anomaly_count_high = ('is_high','sum'),
    earliest_anomaly_ts_high = ('ts_high','min'),
    anomaly_count_low = ('is_low','sum'),
    earliest_anomaly_ts_low = ('ts_low','min')
).reset_index()

# Format earliest anomaly timestamps as strings in UTC+8, keep NaN as <NA>
for col in ['earliest_anomaly_ts_high','earliest_anomaly_ts_low']:
    trace_detail_df[col] = pd.to_datetime(trace_detail_df[col]).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    trace_detail_df[col] = trace_detail_df[col].where(trace_detail_df[col].notna(), pd.NA)

# Also compute anomaly_total per trace_name
trace_detail_df['anomaly_total'] = trace_detail_df['anomaly_count_high'] + trace_detail_df['anomaly_count_low']

# 4) Aggregate per cmdb_id
# total_trace_kpis: distinct trace_name in full trace.csv (global)
total_trace_kpis = trace_df.groupby('cmdb_id')['trace_name'].nunique().rename('total_trace_kpis').reset_index()

# Determine trace kpis with anomaly (anomaly_total>0) from trace_detail_df
trace_with_anom = trace_detail_df[trace_detail_df['anomaly_total'] > 0].copy()

# earliest trace anomaly ts per trace_name (take whichever earliest of high/low)
def earliest_ts_row(r):
    # prefer earliest_anomaly_ts_high if present else low; both strings -> parse
    candidates = []
    for col in ['earliest_anomaly_ts_high','earliest_anomaly_ts_low']:
        v = r.get(col)
        if pd.notna(v):
            candidates.append(pd.to_datetime(v))
    if not candidates:
        return pd.NaT
    return min(candidates)

# Parse earliest per row
trace_with_anom['_earliest_parsed'] = pd.to_datetime(trace_with_anom['earliest_anomaly_ts_high'].fillna(trace_with_anom['earliest_anomaly_ts_low']), errors='coerce')

# Aggregate per cmdb_id
agg_list = []
for cm in top_cmdbs:
    subset_all = trace_detail_df[trace_detail_df['cmdb_id'] == cm]
    subset_anom = trace_with_anom[trace_with_anom['cmdb_id'] == cm]
    total_kpis = int(total_trace_kpis[total_trace_kpis['cmdb_id'] == cm]['total_trace_kpis'].iloc[0]) if cm in total_trace_kpis['cmdb_id'].values else 0
    num_kpis_with_anom = int(subset_anom['trace_name'].nunique())
    # earliest trace anomaly ts across its severe trace rows
    if subset_anom.shape[0] > 0:
        # compute earliest among parsed strings
        parsed = pd.to_datetime(subset_anom['earliest_anomaly_ts_high'].fillna(subset_anom['earliest_anomaly_ts_low']), errors='coerce')
        parsed = parsed.dt.tz_localize(tz, ambiguous='NaT', nonexistent='NaT') if parsed.dt.tz is None else parsed.dt.tz_convert(tz)
        earliest_ts = parsed.min()
        earliest_ts_str = earliest_ts.strftime('%Y-%m-%d %H:%M:%S %Z')
        # sample trace kpis with anomaly: choose up to 5 by earliest anomaly timestamp
        sample_rows = subset_anom.copy()
        # compute parsed ts properly
        sample_rows['_parsed'] = pd.to_datetime(sample_rows['earliest_anomaly_ts_high'].fillna(sample_rows['earliest_anomaly_ts_low']), errors='coerce')
        sample_rows['_parsed'] = sample_rows['_parsed'].dt.tz_localize(tz, ambiguous='NaT', nonexistent='NaT') if sample_rows['_parsed'].dt.tz is None else sample_rows['_parsed'].dt.tz_convert(tz)
        sample_rows = sample_rows.sort_values('_parsed').head(5)
        sample_kpis = "; ".join([
            f"{r['trace_name']} ({pd.to_datetime(r['_parsed']).tz_convert(tz).strftime('%H:%M:%S %Z')})"
            for _, r in sample_rows.iterrows()
        ])
    else:
        earliest_ts_str = pd.NA
        sample_kpis = pd.NA

    total_anomaly_kpis = int(subset_anom['anomaly_total'].sum()) if subset_anom.shape[0] > 0 else 0

    agg_list.append({
        'cmdb_id': cm,
        'total_trace_kpis': total_kpis,
        'num_trace_kpis_with_anomaly': num_kpis_with_anom,
        'earliest_trace_anomaly_ts': earliest_ts_str,
        'sample_trace_kpis_with_anomaly': sample_kpis,
        'total_anomaly_instances': total_anomaly_kpis
    })

trace_candidates = pd.DataFrame(agg_list)

# 5) Sort by num_trace_kpis_with_anomaly desc and return top 10 (already limited to top_cmdbs)
trace_candidates = trace_candidates.sort_values(by='num_trace_kpis_with_anomaly', ascending=False).reset_index(drop=True)
trace_candidates_top10 = trace_candidates.head(10)

# One-line summary: how many of the top-10 had at least one trace KPI anomaly
num_top_with_anom = int((trace_candidates_top10['num_trace_kpis_with_anomaly'] > 0).sum())
summary_line = f"{len(trace_candidates_top10)} top components considered; {num_top_with_anom} had at least one trace KPI anomaly in the incident window."

# Keep trace_detail_df and trace_candidates in kernel for next log inspection
trace_detail_df, trace_candidates_top10, summary_line

```
Out[5]:
```
Summary (plain English, compact):

- I examined trace anomalies for the top-10 components from component_candidates (analysis limited to the 2022-03-20 19:00–19:29:59 UTC+8 window). The detailed per-(cmdb_id,trace_name) table contains 112 trace KPI rows for those components.

- Aggregated findings (top components by number of trace KPIs with anomalies):
  - cartservice-0: total_trace_kpis=28, num_trace_kpis_with_anomaly=28, earliest trace anomaly 2022-03-20 19:00:00, total_anomaly_instances=1130.
  - cartservice-1: total_trace_kpis=28, num_trace_kpis_with_anomaly=28, earliest 19:00:00, total_anomaly_instances=1116.
  - shippingservice-1: total_trace_kpis=24, num_trace_kpis_with_anomaly=24, earliest 19:00:00, total_anomaly_instances=458.
  - shippingservice-2: total_trace_kpis=24, num_trace_kpis_with_anomaly=24, earliest 19:00:00, total_anomaly_instances=507.
  - adservice2-0: total_trace_kpis=4, num_trace_kpis_with_anomaly=4, earliest 19:00:00, total_anomaly_instances=74.
  - paymentservice2-0: total_trace_kpis=4, num_trace_kpis_with_anomaly=4, earliest 19:00:00, total_anomaly_instances=80.
  - node-2, node-1, node-6, node-5: no trace KPI anomalies detected in the window (num_trace_kpis_with_anomaly=0).

- Top anomalous trace patterns: many “from_frontend” and “from_checkoutservice” trace metrics show repeated anomalies (some trace KPIs have anomaly_total up to 60), and earliest anomalies appear at or very near 19:00:00 UTC+8.

- One-line summary: Of the 10 top components considered, 6 had at least one trace KPI anomaly in the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                  trace_name  total_points_in_window  anomaly_count_high earliest_anomaly_ts_high  anomaly_count_low  earliest_anomaly_ts_low  anomaly_total
0         adservice2-0        trace.from_frontend2-0.duration_mean                      30                   2  2022-03-20 19:06:00 CST                  4  2022-03-20 19:09:00 CST              6
1         adservice2-0         trace.from_frontend2-0.duration_p95                      30                   2  2022-03-20 19:06:00 CST                  2  2022-03-20 19:13:00 CST              4
2         adservice2-0           trace.from_frontend2-0.error_rate                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
3         adservice2-0            trace.from_frontend2-0.row_count                      30                   2  2022-03-20 19:09:00 CST                  2  2022-03-20 19:14:00 CST              4
4        cartservice-0  trace.from_checkoutservice-0.duration_mean                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
..                 ...                                         ...                     ...                 ...                      ...                ...                      ...            ...
107  shippingservice-2             trace.from_frontend-1.row_count                      30                   3  2022-03-20 19:01:00 CST                  2  2022-03-20 19:02:00 CST              5
108  shippingservice-2         trace.from_frontend-2.duration_mean                      30                  10  2022-03-20 19:00:00 CST                  3  2022-03-20 19:06:00 CST             13
109  shippingservice-2          trace.from_frontend-2.duration_p95                      30                   3  2022-03-20 19:05:00 CST                  2  2022-03-20 19:07:00 CST              5
110  shippingservice-2            trace.from_frontend-2.error_rate                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
111  shippingservice-2             trace.from_frontend-2.row_count                      30                   2  2022-03-20 19:05:00 CST                  2  2022-03-20 19:26:00 CST              4

[112 rows x 8 columns],              cmdb_id  total_trace_kpis  num_trace_kpis_with_anomaly earliest_trace_anomaly_ts                     sample_trace_kpis_with_anomaly  total_anomaly_instances
0      cartservice-0                28                           28   2022-03-20 19:00:00 CST  trace.from_checkoutservice-0.duration_mean (19...                     1130
1      cartservice-1                28                           28   2022-03-20 19:00:00 CST  trace.from_checkoutservice-0.duration_mean (19...                     1116
2  shippingservice-1                24                           24   2022-03-20 19:00:00 CST  trace.from_checkoutservice-0.duration_mean (19...                      458
3  shippingservice-2                24                           24   2022-03-20 19:00:00 CST  trace.from_checkoutservice-0.error_rate (19:00...                      507
4       adservice2-0                 4                            4   2022-03-20 19:00:00 CST  trace.from_frontend2-0.error_rate (19:00:00 CS...                       74
5  paymentservice2-0                 4                            4   2022-03-20 19:00:00 CST  trace.from_checkoutservice2-0.error_rate (19:0...                       80
6             node-2                 0                            0                      <NA>                                               <NA>                        0
7             node-1                 0                            0                      <NA>                                               <NA>                        0
8             node-6                 0                            0                      <NA>                                               <NA>                        0
9             node-5                 0                            0                      <NA>                                               <NA>                        0, '10 top components considered; 6 had at least one trace KPI anomaly in the incident window.')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_df from kernel (loaded previously)
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Ensure ts column is timezone-aware Asia/Shanghai
if 'ts' not in trace_df.columns:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware & converted
    trace_df['ts'] = pd.to_datetime(trace_df['ts']).dt.tz_convert(tz)

# Target components
carts = ['cartservice-0', 'cartservice-1']

# 1) Compute global P95 and P5 for (cmdb_id, trace_name) using full series (no time filtering)
trace_thresholds_cart = (
    trace_df[trace_df['cmdb_id'].isin(carts)]
    .groupby(['cmdb_id', 'trace_name'])['value']
    .quantile([0.95, 0.05])
    .unstack(level=-1)
    .rename(columns={0.95: 'global_P95', 0.05: 'global_P5'})
    .reset_index()
)

# 2) Filter trace.csv to incident window and to the two carts
start_dt = tz.localize(datetime(2022, 3, 20, 19, 0, 0))
end_dt   = tz.localize(datetime(2022, 3, 20, 19, 30, 0))  # exclusive
window_cart = trace_df[
    (trace_df['cmdb_id'].isin(carts)) &
    (trace_df['ts'] >= start_dt) & (trace_df['ts'] < end_dt)
].copy()

# 3) Merge thresholds and compute anomalies per (cmdb_id, trace_name)
window_cart = window_cart.merge(trace_thresholds_cart, on=['cmdb_id', 'trace_name'], how='left')

window_cart['is_high'] = window_cart['value'] >= window_cart['global_P95']
window_cart['is_low']  = window_cart['value'] <= window_cart['global_P5']
window_cart['ts_high'] = window_cart['ts'].where(window_cart['is_high'])
window_cart['ts_low']  = window_cart['ts'].where(window_cart['is_low'])

agg_cart = window_cart.groupby(['cmdb_id', 'trace_name']).agg(
    total_points_in_window = ('value', 'size'),
    anomaly_count_high = ('is_high', 'sum'),
    earliest_anomaly_ts_high = ('ts_high', 'min'),
    anomaly_count_low = ('is_low', 'sum'),
    earliest_anomaly_ts_low = ('ts_low', 'min')
).reset_index()

# 4) Compute anomaly_total and format earliest timestamps as strings (UTC+8)
agg_cart['anomaly_total'] = agg_cart['anomaly_count_high'] + agg_cart['anomaly_count_low']

def fmt_ts(s):
    if pd.isna(s):
        return pd.NA
    try:
        return pd.to_datetime(s).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    except Exception:
        return pd.NA

agg_cart['earliest_anomaly_ts_high'] = agg_cart['earliest_anomaly_ts_high'].apply(fmt_ts)
agg_cart['earliest_anomaly_ts_low']  = agg_cart['earliest_anomaly_ts_low'].apply(fmt_ts)

# Store detailed per-(cmdb_id,trace_name) table in kernel variable as requested
cart_trace_details = agg_cart[[
    'cmdb_id', 'trace_name', 'total_points_in_window',
    'anomaly_count_high', 'earliest_anomaly_ts_high',
    'anomaly_count_low', 'earliest_anomaly_ts_low',
    'anomaly_total'
]].sort_values(by='anomaly_total', ascending=False).reset_index(drop=True)

# Top 20 rows for compact return
cart_trace_top20 = cart_trace_details.head(20)

# Summary: earliest anomaly timestamp across both carts and count of distinct trace_name with anomaly_total>0
anom_rows = agg_cart[agg_cart['anomaly_total'] > 0].copy()
if anom_rows.shape[0] > 0:
    # compute per-row earliest (min of high/low) using original tz-aware columns from window_cart aggregation
    # need to recompute parsed earliest from the window_cart grouped values
    parsed_high = pd.to_datetime(agg_cart['earliest_anomaly_ts_high'], errors='coerce')
    parsed_low  = pd.to_datetime(agg_cart['earliest_anomaly_ts_low'], errors='coerce')
    # convert to tz-aware where possible (strings may be NaT)
    # fallback: use window_cart grouped min directly to avoid parsing issues; recompute here:
    agg_cart_raw = window_cart.groupby(['cmdb_id','trace_name']).agg(
        earliest_high_raw = ('ts_high','min'),
        earliest_low_raw = ('ts_low','min')
    ).reset_index()
    merged_for_earliest = agg_cart.merge(agg_cart_raw, on=['cmdb_id','trace_name'], how='left')
    # row-wise earliest
    row_earliest = merged_for_earliest[['earliest_high_raw','earliest_low_raw']].min(axis=1)
    # restrict to anomaly_total>0 rows
    row_earliest = row_earliest[merged_for_earliest['anomaly_total'] > 0]
    overall_earliest = row_earliest.min()
    if pd.isna(overall_earliest):
        earliest_str = pd.NA
    else:
        earliest_str = pd.to_datetime(overall_earliest).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    distinct_trace_names_with_anom = int(anom_rows['trace_name'].nunique())
else:
    earliest_str = pd.NA
    distinct_trace_names_with_anom = 0

summary_line = f"Earliest anomaly across cartservice-0/1: {earliest_str}; distinct trace_name with anomaly_total>0: {distinct_trace_names_with_anom}."

# Return compact outputs (top 20 table and summary), and keep full cart_trace_details in kernel
cart_trace_top20, summary_line

```
Out[6]:
```
Summary (plain English, compact):

- For cartservice-0 and cartservice-1 the top per-trace anomalies are dominated by checkoutservice- and frontend-origin traces. Many trace KPIs show full-window anomalies: total_points_in_window=30 and anomaly_total=60 (30 high + 30 low) for many traces.

- Top examples (from the returned top-20):
  - cartservice-0 · trace.from_checkoutservice-0.duration_mean — 30 points, anomaly_count_high=30, anomaly_count_low=30, anomaly_total=60.
  - cartservice-0 · trace.from_checkoutservice-0.duration_p95 — 30 points, anomaly_total=60.
  - cartservice-0 · trace.from_checkoutservice-0.error_rate — 30 points, anomaly_total=60.
  - cartservice-1 · trace.from_checkoutservice-2.duration_mean / duration_p95 / error_rate — each 30 points, anomaly_total=60.

- Earliest anomaly across both pods: 2022-03-20 19:00:00 UTC+8.
- Distinct trace_name entries with anomaly_total > 0 across cartservice-0 and cartservice-1: 28.

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id                                  trace_name  total_points_in_window  anomaly_count_high earliest_anomaly_ts_high  anomaly_count_low  earliest_anomaly_ts_low  anomaly_total
0   cartservice-0  trace.from_checkoutservice-0.duration_mean                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
1   cartservice-0   trace.from_checkoutservice-0.duration_p95                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
2   cartservice-0     trace.from_checkoutservice-0.error_rate                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
3   cartservice-0  trace.from_checkoutservice-1.duration_mean                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
4   cartservice-0     trace.from_checkoutservice-1.error_rate                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
5   cartservice-0   trace.from_checkoutservice-1.duration_p95                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
6   cartservice-0   trace.from_checkoutservice-2.duration_p95                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
7   cartservice-0  trace.from_checkoutservice-2.duration_mean                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
8   cartservice-0            trace.from_frontend-0.error_rate                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
9   cartservice-0            trace.from_frontend-1.error_rate                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
10  cartservice-0     trace.from_checkoutservice-2.error_rate                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
11  cartservice-0          trace.from_frontend-0.duration_p95                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
12  cartservice-0            trace.from_frontend-2.error_rate                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
13  cartservice-1  trace.from_checkoutservice-2.duration_mean                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
14  cartservice-1   trace.from_checkoutservice-2.duration_p95                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
15  cartservice-1     trace.from_checkoutservice-2.error_rate                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
16  cartservice-1  trace.from_checkoutservice-1.duration_mean                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
17  cartservice-1   trace.from_checkoutservice-1.duration_p95                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
18  cartservice-1     trace.from_checkoutservice-1.error_rate                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
19  cartservice-1     trace.from_checkoutservice-0.error_rate                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60, 'Earliest anomaly across cartservice-0/1: 2022-03-20 19:00:00 CST; distinct trace_name with anomaly_total>0: 28.')```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')
carts = ['cartservice-0', 'cartservice-1']

# Reuse log_df and error_df from kernel if present, otherwise load
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

try:
    error_df
except NameError:
    error_df = pd.read_csv('error_logs.csv')

# Ensure timezone-aware timestamp columns for both DataFrames
for df in (log_df, error_df):
    if 'ts' not in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df['ts'] = pd.to_datetime(df['ts']).dt.tz_convert(tz)

# Part 1: log.csv analysis for cartservice-0 and cartservice-1

# 1) Compute global P95 and P5 for (cmdb_id, log_name) using full series (only for target cmdb_ids)
logs_target = log_df[log_df['cmdb_id'].isin(carts)].copy()
if logs_target.shape[0] > 0:
    grp = logs_target.groupby(['cmdb_id', 'log_name'])['value']
    global_p95 = grp.quantile(0.95).rename('global_P95')
    global_p5  = grp.quantile(0.05).rename('global_P5')
    log_thresholds = pd.concat([global_p95, global_p5], axis=1).reset_index()
else:
    log_thresholds = pd.DataFrame(columns=['cmdb_id','log_name','global_P95','global_P5'])

# 2) Filter log.csv to the incident window and target cmdb_ids
start_dt = tz.localize(datetime(2022,3,20,19,0,0))
end_dt   = tz.localize(datetime(2022,3,20,19,30,0))  # exclusive upper bound
window_logs = log_df[
    (log_df['cmdb_id'].isin(carts)) &
    (log_df['ts'] >= start_dt) & (log_df['ts'] < end_dt)
].copy()

# 3) Merge thresholds, compute anomalies and aggregate per (cmdb_id, log_name)
if window_logs.shape[0] > 0:
    merged_logs = window_logs.merge(log_thresholds, on=['cmdb_id','log_name'], how='left')
    merged_logs['is_high'] = merged_logs['value'] >= merged_logs['global_P95']
    merged_logs['is_low']  = merged_logs['value'] <= merged_logs['global_P5']
    merged_logs['ts_high'] = merged_logs['ts'].where(merged_logs['is_high'])
    merged_logs['ts_low']  = merged_logs['ts'].where(merged_logs['is_low'])

    cart_log_details = merged_logs.groupby(['cmdb_id','log_name']).agg(
        global_P95 = ('global_P95','first'),
        global_P5  = ('global_P5','first'),
        total_points_in_window = ('value','size'),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min'),
        anomaly_count_high = ('is_high','sum'),
        earliest_anomaly_ts_high = ('ts_high','min'),
        anomaly_count_low = ('is_low','sum'),
        earliest_anomaly_ts_low = ('ts_low','min')
    ).reset_index()
else:
    # empty structure if no rows in window
    cart_log_details = pd.DataFrame(columns=[
        'cmdb_id','log_name','global_P95','global_P5','total_points_in_window',
        'max_value_in_window','min_value_in_window',
        'anomaly_count_high','earliest_anomaly_ts_high',
        'anomaly_count_low','earliest_anomaly_ts_low'
    ])

# Format earliest timestamps as compact strings and replace NaT with <NA>
for col in ['earliest_anomaly_ts_high','earliest_anomaly_ts_low']:
    if col in cart_log_details.columns:
        cart_log_details[col] = pd.to_datetime(cart_log_details[col]).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        cart_log_details[col] = cart_log_details[col].where(cart_log_details[col].notna(), pd.NA)

# Compute anomaly_total for sorting and select requested columns
if 'anomaly_count_high' in cart_log_details.columns and 'anomaly_count_low' in cart_log_details.columns:
    cart_log_details['anomaly_total'] = cart_log_details['anomaly_count_high'] + cart_log_details['anomaly_count_low']
else:
    cart_log_details['anomaly_total'] = 0

# Top 10 rows sorted by anomaly_total desc (compact)
cart_log_details_top10 = cart_log_details.sort_values(by='anomaly_total', ascending=False)[
    ['cmdb_id','log_name','global_P95','global_P5','total_points_in_window',
     'anomaly_count_high','earliest_anomaly_ts_high','anomaly_count_low','earliest_anomaly_ts_low',
     'max_value_in_window','min_value_in_window']
].head(10).reset_index(drop=True)

# Part 2: error_logs.csv check for cartservice-0 and cartservice-1
err_target = error_df[error_df['cmdb_id'].isin(carts)].copy()

if err_target.shape[0] > 0:
    # For each cart compute count and up to 5 sample messages
    def sample_msgs(sub):
        samples = sub['message'].dropna().astype(str).unique()[:5].tolist()
        return "; ".join(samples)
    cart_err_summary = err_target.groupby('cmdb_id').agg(
        error_count = ('message','count'),
        sample_messages = (lambda s: "; ".join(s.dropna().astype(str).unique()[:5]),)
    ).reset_index()
    # Normalize column names if needed
    cart_errorlog_summary = cart_err_summary.rename(columns={0:'sample_messages'}) if 'sample_messages' not in cart_err_summary.columns else cart_err_summary
else:
    # No error logs for carts: return distinct cmdb_ids present in error_df (up to 10) and a 1-line sample message
    distinct_ids = error_df['cmdb_id'].dropna().unique()[:10].tolist()
    sample_msg = error_df['message'].dropna().astype(str).unique()
    sample_msg_one = sample_msg[0] if len(sample_msg) > 0 else pd.NA
    cart_errorlog_summary = pd.DataFrame({
        'note': ['no error_logs for cartservice-0/1'],
        'distinct_errorlog_cmdb_ids_up_to_10': [", ".join(distinct_ids)],
        'sample_message': [sample_msg_one]
    })

# Expose cart_log_details and cart_errorlog_summary in kernel (already assigned)
cart_log_details, cart_log_details_top10, cart_errorlog_summary

```
Out[7]:
```
```python
# Compose a compact plain-English summary of the cartservice log & error-log analysis
try:
    cart_log = cart_log_details_top10  # from kernel
except NameError:
    cart_log = None

try:
    cart_err = cart_errorlog_summary  # from kernel
except NameError:
    cart_err = None

summary_lines = []

if cart_log is None or cart_log.empty:
    summary_lines.append("No log entries for cartservice-0/1 found in the incident window.")
else:
    summary_lines.append("Log analysis for cartservice-0 and cartservice-1 (2022-03-20 19:00–19:29:59 UTC+8):")
    # Group by cmdb_id and summarize key log_name findings compactly
    for cm, grp in cart_log.groupby('cmdb_id'):
        summary_lines.append(f"- {cm}:")
        for _, r in grp.iterrows():
            ln = r['log_name']
            high = int(r['anomaly_count_high']) if pd.notna(r['anomaly_count_high']) else 0
            low  = int(r['anomaly_count_low'])  if pd.notna(r['anomaly_count_low']) else 0
            tp   = int(r['total_points_in_window']) if pd.notna(r['total_points_in_window']) else 0
            earliest_h = r['earliest_anomaly_ts_high'] if pd.notna(r['earliest_anomaly_ts_high']) else "<NA>"
            earliest_l = r['earliest_anomaly_ts_low']  if pd.notna(r['earliest_anomaly_ts_low']) else "<NA>"
            maxv = r['max_value_in_window'] if 'max_value_in_window' in r and pd.notna(r['max_value_in_window']) else "<NA>"
            minv = r['min_value_in_window'] if 'min_value_in_window' in r and pd.notna(r['min_value_in_window']) else "<NA>"
            summary_lines.append(
                f"    • {ln}: {tp} points; anomaly_count_high={high} (earliest={earliest_h}); "
                f"anomaly_count_low={low} (earliest={earliest_l}); max={maxv}; min={minv}."
            )

# Error logs summary
if cart_err is None:
    summary_lines.append("Error log data not available.")
else:
    # Detect "no error_logs for cartservice-0/1" case
    if 'note' in cart_err.columns and cart_err['note'].str.contains('no error_logs').any():
        distinct_ids = cart_err['distinct_errorlog_cmdb_ids_up_to_10'].iloc[0] if 'distinct_errorlog_cmdb_ids_up_to_10' in cart_err.columns else "<none>"
        sample_msg = cart_err['sample_message'].iloc[0] if 'sample_message' in cart_err.columns else "<none>"
        summary_lines.append("Error logs: no entries for cartservice-0 or cartservice-1.")
        summary_lines.append(f"  Distinct cmdb_id values present in error_logs.csv (up to 10): {distinct_ids}.")
        summary_lines.append(f"  Sample message from error_logs: {sample_msg}")
    else:
        # Per-cart counts and samples
        summary_lines.append("Error logs for carts:")
        # cart_err may have columns ['cmdb_id','error_count','sample_messages'] or similar
        for _, row in cart_err.iterrows():
            cid = row.get('cmdb_id', '<unknown>')
            cnt = int(row.get('error_count', 0)) if pd.notna(row.get('error_count', None)) else 0
            samples = row.get('sample_messages', row.get('sample_messages', pd.NA))
            summary_lines.append(f"  - {cid}: {cnt} rows; sample messages: {samples}")

# Join into a single compact summary string
summary = " ".join(summary_lines)

# Display the summary (variable output)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id         log_name  global_P95  global_P5  total_points_in_window  max_value_in_window  min_value_in_window  anomaly_count_high earliest_anomaly_ts_high  anomaly_count_low  earliest_anomaly_ts_low  anomaly_total
0  cartservice-0  log.error_count        0.00        0.0                      30                  0.0                  0.0                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
1  cartservice-0    log.row_count     1343.55     1196.9                      30               1366.0               1191.0                   2  2022-03-20 19:01:00 CST                  2  2022-03-20 19:07:00 CST              4
2  cartservice-1  log.error_count        0.00        0.0                      30                  0.0                  0.0                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST             60
3  cartservice-1    log.row_count     1340.00     1213.1                      30               1360.0               1190.0                   2  2022-03-20 19:01:00 CST                  2  2022-03-20 19:10:00 CST              4,          cmdb_id         log_name  global_P95  global_P5  total_points_in_window  anomaly_count_high earliest_anomaly_ts_high  anomaly_count_low  earliest_anomaly_ts_low  max_value_in_window  min_value_in_window
0  cartservice-0  log.error_count        0.00        0.0                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST                  0.0                  0.0
1  cartservice-1  log.error_count        0.00        0.0                      30                  30  2022-03-20 19:00:00 CST                 30  2022-03-20 19:00:00 CST                  0.0                  0.0
2  cartservice-0    log.row_count     1343.55     1196.9                      30                   2  2022-03-20 19:01:00 CST                  2  2022-03-20 19:07:00 CST               1366.0               1191.0
3  cartservice-1    log.row_count     1340.00     1213.1                      30                   2  2022-03-20 19:01:00 CST                  2  2022-03-20 19:10:00 CST               1360.0               1190.0,                                 note distinct_errorlog_cmdb_ids_up_to_10                                     sample_message
0  no error_logs for cartservice-0/1  frontend-0, frontend-2, frontend-1  severity: warning, message: failed to retrieve...)```
```